In [1]:
import yfinance as yf
import requests
from bs4 import BeautifulSoup
import json

In [8]:
url = 'https://www.onvista.de/suche?searchValue={}'

In [30]:
res = requests.get(url.format('US0378331005'))
res.status_code

200

In [32]:
# view with beautiful soup
soup = BeautifulSoup(res.text, 'html.parser')
# print(soup.prettify())

In [33]:
# find script tag with data
script = soup.find_all('script', {'id': '__NEXT_DATA__'})

In [35]:
obj = json.loads(script[0].text)

In [29]:
obj

{'props': {'pageProps': {'searchValue': 'aapl'}, '__N_SSP': True},
 'page': '/suche',
 'query': {'searchValue': 'aapl'},
 'buildId': 'ro5boekuT0h0O4A7r3-vK',
 'isFallback': False,
 'dynamicIds': [7446, 26597],
 'gssp': True,
 'scriptLoader': []}

In [20]:
import os
from pathlib import Path
from google import genai
from google.genai import types
from pydantic import BaseModel
import subprocess
import os

def load_sh_env(script_path):
    # Run the script, then output the environment
    command = f"source {script_path} && env"
    result = subprocess.run(['bash', '-c', command], capture_output=True, text=True)
    
    for line in result.stdout.splitlines():
        if '=' in line:
            key, value = line.split('=', 1)
            os.environ[key] = value

load_sh_env('~/additional_env_vars.sh')

In [21]:
'GEMINI_API_KEY' in os.environ

True

In [22]:
class CityInfo(BaseModel):
    name: str
    country: str
    population: int
    
client = genai.Client()

In [23]:

# 3. Call the Gemma 4 model and apply the schema constraint
response = client.models.generate_content(
    model='gemma-4-26b-a4b-it',
    contents='Give me basic information about your favorite city in the World.',
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=CityInfo,
    )
)

In [25]:
response.parsed

CityInfo(name='Tokyo', country='Japan', population=13960000)

In [28]:
response.usage_metadata

GenerateContentResponseUsageMetadata(
  candidates_token_count=25,
  prompt_token_count=13,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=13
    ),
  ],
  total_token_count=38
)

In [34]:
# list available models
models = list(client.models.list())
for model in models:
	if 'gemma' in model.name.lower():
		print(model.name)

models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
